# ✈️ Airline Passenger Satisfaction — SPRINT 3
## Optimization & Final Model
**Goal:** Improve model performance and reliability using feature engineering, selection, and hyperparameter tuning

---
## 🔁 Setup — Load Data & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, ConfusionMatrixDisplay, roc_auc_score
)

# ---------- Load & Clean ----------
df = pd.read_csv("../data/airline_passenger_satisfaction.csv")
imputer = SimpleImputer(strategy='median')
df["Arrival Delay"] = imputer.fit_transform(df[["Arrival Delay"]])
df.drop_duplicates(inplace=True)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")
if 'unnamed:_0' in df.columns:
    df.drop('unnamed:_0', axis=1, inplace=True)
for col in ['departure_delay', 'arrival_delay']:
    Q1 = df[col].quantile(0.25); Q3 = df[col].quantile(0.75); IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

print(f"✅ Dataset loaded: {df.shape}")

---
## 📌 Step 1 — Feature Engineering

In [ ]:
# Create new meaningful features

# 1. Total delay
df['total_delay'] = df['departure_delay'] + df['arrival_delay']

# 2. Delay ratio (arrival vs departure)
df['delay_ratio'] = df['arrival_delay'] / (df['departure_delay'] + 1)

# 3. Average service rating (all service columns)
service_cols = [
    'departure_and_arrival_time_convenience', 'ease_of_online_booking',
    'check_in_service', 'online_boarding', 'gate_location',
    'on_board_service', 'seat_comfort', 'leg_room_service',
    'cleanliness', 'food_and_drink', 'in_flight_service',
    'in_flight_wifi_service', 'in_flight_entertainment', 'baggage_handling'
]
df['avg_service_rating'] = df[service_cols].mean(axis=1)

# 4. In-flight experience score (comfort + entertainment + service)
df['inflight_experience'] = (
    df['seat_comfort'] + df['in_flight_entertainment'] +
    df['in_flight_service'] + df['in_flight_wifi_service']
) / 4

# 5. Ground experience score
df['ground_experience'] = (
    df['check_in_service'] + df['gate_location'] + df['baggage_handling'] +
    df['ease_of_online_booking']
) / 4

# 6. Is long-haul flight
df['is_long_haul'] = (df['flight_distance'] > df['flight_distance'].median()).astype(int)

print("✅ New features created:")
new_features = ['total_delay', 'delay_ratio', 'avg_service_rating',
                'inflight_experience', 'ground_experience', 'is_long_haul']
for f in new_features:
    print(f"  {f}: min={df[f].min():.2f}, max={df[f].max():.2f}, mean={df[f].mean():.2f}")

---
## 📌 Step 2 — Feature Selection

In [ ]:
# Encode target
le_y = LabelEncoder()
y_enc = le_y.fit_transform(df['satisfaction'])

# Updated feature sets with engineered features
cat_cols = ['gender', 'customer_type', 'type_of_travel', 'class']
num_cols = [
    'age', 'flight_distance', 'departure_delay', 'arrival_delay',
    'departure_and_arrival_time_convenience', 'ease_of_online_booking',
    'check_in_service', 'online_boarding', 'gate_location',
    'on_board_service', 'seat_comfort', 'leg_room_service',
    'cleanliness', 'food_and_drink', 'in_flight_service',
    'in_flight_wifi_service', 'in_flight_entertainment', 'baggage_handling',
    # Engineered features
    'total_delay', 'delay_ratio', 'avg_service_rating',
    'inflight_experience', 'ground_experience', 'is_long_haul'
]

X = df[cat_cols + num_cols]

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"Shape after feature engineering: {X_train_proc.shape}")

# Feature importance using Random Forest for selection
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_train_proc, y_train)

feature_names = preprocessor.get_feature_names_out()
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_selector.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 20 Important Features:")
display(feat_imp_df.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(data=feat_imp_df.head(20), x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance after Engineering — Top 20')
plt.tight_layout()
plt.savefig('../data/sprint3_feature_importance.png', dpi=100)
plt.show()

In [ ]:
# Correlation matrix of engineered features with target
corr_cols = num_cols + ['satisfaction_enc']
df_corr = df[num_cols].copy()
df_corr['satisfaction_enc'] = y_enc[:len(df_corr)] if len(df_corr) == len(y_enc) else y_enc

target_corr = df_corr.corr()['satisfaction_enc'].drop('satisfaction_enc').abs().sort_values(ascending=False)
print("\nFeature Correlation with Target:")
print(target_corr.head(15).to_string())

---
## 📌 Step 3 — Hyperparameter Tuning

### 3a. Random Forest — GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth'   : [None, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train_proc, y_train)

print(f"✅ RF Best Params: {rf_grid.best_params_}")
print(f"   Best CV F1    : {rf_grid.best_score_:.4f}")
best_rf = rf_grid.best_estimator_

### 3b. Gradient Boosting — RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

gb_param_dist = {
    'n_estimators'  : [100, 200, 300],
    'learning_rate' : [0.05, 0.1, 0.2],
    'max_depth'     : [3, 4, 5, 6],
    'subsample'     : [0.8, 0.9, 1.0],
    'min_samples_split': [2, 5, 10]
}

gb_random = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=gb_param_dist,
    n_iter=20, cv=5, scoring='f1', n_jobs=-1, random_state=42, verbose=1
)
gb_random.fit(X_train_proc, y_train)

print(f"✅ GB Best Params : {gb_random.best_params_}")
print(f"   Best CV F1     : {gb_random.best_score_:.4f}")
best_gb = gb_random.best_estimator_

### 3c. Logistic Regression — GridSearchCV

In [ ]:
lr_param_grid = {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid=lr_param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train_proc, y_train)

print(f"✅ LR Best Params : {lr_grid.best_params_}")
print(f"   Best CV F1     : {lr_grid.best_score_:.4f}")
best_lr = lr_grid.best_estimator_

---
## 📌 Step 4 — Build Final Model (Best Estimator)

In [ ]:
# Compare tuned models
tuned_models = {
    'Random Forest (Tuned)'      : best_rf,
    'Gradient Boosting (Tuned)'  : best_gb,
    'Logistic Regression (Tuned)': best_lr,
}

tuned_results = []
for name, model in tuned_models.items():
    y_pred = model.predict(X_test_proc)
    tuned_results.append({
        'Model'    : name,
        'Test Acc' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1 Score' : round(f1_score(y_test, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_test, model.predict_proba(X_test_proc)[:,1]), 4)
    })

tuned_df = pd.DataFrame(tuned_results).sort_values('F1 Score', ascending=False).reset_index(drop=True)
print("Tuned Model Comparison:")
display(tuned_df)

# Select best model
best_model_name = tuned_df.iloc[0]['Model']
best_model = tuned_models[best_model_name]
print(f"\n🏆 Best Final Model: {best_model_name}")

In [ ]:
# Build final end-to-end pipeline
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', best_model)
])

# Re-train on unprocessed X (pipeline handles preprocessing internally)
final_pipeline.fit(X_train, y_train)

y_pred_final = final_pipeline.predict(X_test)
print("=" * 55)
print("FINAL MODEL EVALUATION")
print("=" * 55)
print(f"Model    : {best_model_name}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_final):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, final_pipeline.predict_proba(X_test)[:,1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=le_y.classes_))

fig, ax = plt.subplots(figsize=(6, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=le_y.classes_, cmap='Blues', ax=ax
)
ax.set_title(f'Final Model — {best_model_name}')
plt.tight_layout()
plt.savefig('../data/final_model_cm.png', dpi=100)
plt.show()

---
## 📌 Step 5 — Final Evaluation on Unseen Test Data

In [ ]:
from sklearn.model_selection import learning_curve

# Learning curve
train_sizes, train_scores, test_scores = learning_curve(
    final_pipeline, X_train, y_train,
    cv=5, scoring='f1', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8)
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='steelblue', label='Train F1')
plt.plot(train_sizes, test_scores.mean(axis=1),  'o-', color='coral',     label='CV F1')
plt.fill_between(train_sizes,
    train_scores.mean(1) - train_scores.std(1),
    train_scores.mean(1) + train_scores.std(1), alpha=0.15, color='steelblue')
plt.fill_between(train_sizes,
    test_scores.mean(1) - test_scores.std(1),
    test_scores.mean(1) + test_scores.std(1), alpha=0.15, color='coral')
plt.xlabel('Training Size'); plt.ylabel('F1 Score')
plt.title(f'Learning Curve — {best_model_name}')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/learning_curve.png', dpi=100)
plt.show()

---
## 📌 Step 6 — Model Serialization with Pickle

In [ ]:
import pickle

# Save the complete pipeline (preprocessor + model)
with open('../models/final_model_pipeline.pkl', 'wb') as f:
    pickle.dump(final_pipeline, f)

# Save label encoder
with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le_y, f)

# Save feature metadata
feature_meta = {
    'cat_cols': cat_cols,
    'num_cols': num_cols,
    'service_cols': service_cols,
    'model_name': best_model_name,
    'test_accuracy': accuracy_score(y_test, y_pred_final),
    'test_f1': f1_score(y_test, y_pred_final),
    'classes': list(le_y.classes_)
}
with open('../models/feature_meta.pkl', 'wb') as f:
    pickle.dump(feature_meta, f)

# Verify model loads correctly
with open('../models/final_model_pipeline.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
verify_pred = loaded_model.predict(X_test[:5])

print("=" * 55)
print("SPRINT 3 DELIVERABLES")
print("=" * 55)
print("✔ Final optimized model saved         → models/final_model_pipeline.pkl")
print("✔ Label encoder saved                 → models/label_encoder.pkl")
print("✔ Feature metadata saved              → models/feature_meta.pkl")
print("✔ Feature engineering steps complete")
print(f"\n🔍 Model verification (5 samples): {verify_pred}")
print("\n➡️  Proceed to Sprint 4 → Deployment & MLOps")